# Sudoku: Set-Theoretic Formulation and Strategy-Based Algorithmic Solver

## Abstract ##

This notebook observes the popular puzzle of Sudoku through the lens of set theory and predicate logic. Although the puzzle is defined by three simple constraints, logic and constraint have led to the development of a number of effective strategies, which are essential for cracking more advanced puzzles and obtaining the unique solution. We will take a look at the strategies required to solve most of the solvable puzzles and apply them to a selected sample varying in difficulty. Finally, we will observe an animation of the SUdoku board being solved in real time and a 3D interpretation of the solution path.

## Sudoku basics ##
A Sudoku puzzle is a partially filled $9 \times 9$ grid, for which each row, column and $3 \times 3$ box contains the digits from $1$ to $9$ exactly once. A puzzle in its initial, partially filled state has one unique solution, meaning that the digits can be distributed on the board in one particular way only. Gary McGuire and collaborators proved that no proper Sudoku puzzle with a unique solution can have fewer than 17 clues, settling the long-standing minimum-clue problem in 2012. However, a unique solution does not necessarily mean that a puzzle can be solved by logical deduction alone. Some of the hardest puzzles require backtracking, and we will not be considering those in our algorithm testing.
<br>A Sudoku puzzle can be viewed in terms of candidate sets. For every empty cell, one can consider the set of digits that do not violate the row, column, and box conditions. In this way, a Sudoku puzzle may be viewed not only as a $9 \times 9$ grid of digits, but also as a $9 \times 9$ grid of candidate sets. For a solved or initially given cell, the candidate set consists of the single assigned digit, whereas for an empty cell it contains all values that are still possible. Solving Sudoku then becomes a process of progressively reducing these candidate sets until forced placements can be made. In particular, if a candidate set has size $1$, then the corresponding digit must be placed in that cell.



## Formalizing Sudoku as Sets and Predicate Logic
We can start by defying the set of digits that may appear in a puzzle as
$$ D = \{1,2,...,9\}. $$
A standard sudoku grid of $9 \times 9$ contains $81$ cells, each of them identified by a pair of coordinates $(i,j)$, where i denotes the row index and j the column index. The set of all cells is therefore defined by
$$G = \{(i,j) \ | \ 1\leq i \leq 9,\  1\leq j \leq 9\}.$$
A complete Sudku solution may be viewed as a relation $R \subseteq G \times D$, where $G \times D$ is the Cartesian product of the set of cells $G$ and set of digits $D$. In this formulation, $R$ consists of ordered pairs of the form $((i,j),d)$, where $(i,j) \in G$ and $d \in D$. The interpretaton of $((i,j),d) \in R$ is that the cell $(i,j)$ contains the digit $d$.
<br> Equivalently, one may define a predicate $P(i,j,d)$ by
$$P(i,j,d) \ \mbox{is true if and only if} \ ((i,j),d) \in R.$$
Thus, the predicate $P(i,j,d)$ expresses the statement that digit $d$ is placed in cell $(i,j)$.
<br> In addition to rows and columns, the Sudoku grid is split into nine $3 \times\ 3$ subgrids. For each $k,l \in \{1,2,3\}$ define the subgrid $S_{k,l}$ as
$$ S_{k,l} = \{(i,j) \ | \ 3k-2 \leq i \leq 3k, \ 3l - 2 \leq j \leq 3l\}. $$
<br> A valid Sudoku must satisfy four conditions.
### Row condition
Each row contains any given digit at most once:
$$ \forall i \in \{1,\dots,9\},\;\forall d \in D,\;\forall j_1,j_2 \in \{1,\dots,9\},\;
(j_1 \neq j_2) \Rightarrow \neg\bigl(P(i,j_1,d)\wedge P(i,j_2,d)\bigr) $$
### Column condition
Each column contains any given digit at most once:
$$ \forall j \in \{1,\dots,9\},\;\forall d \in D,\;\forall i_1,i_2 \in \{1,\dots,9\},\;
(i_1 \neq i_2) \Rightarrow \neg\bigl(P(i_1,j,d)\wedge P(i_2,j,d)\bigr) $$
### Subgrid condition
Each subgrid contains any given digit at most once:
$$ \forall k,l \in \{1,2,3\},\;\forall d \in D,\;\forall (i_1,j_1),(i_2,j_2) \in S_{k,l},\;
(i_1,j_1)\neq(i_2,j_2) \Rightarrow \neg\bigl(P(i_1,j_1,d)\wedge P(i_2,j_2,d)\bigr) $$
### Cell condition
Every cell contains exactly one digit
$$ \forall i \in \{1,\ldots,9\},\forall j \in \{1,\ldots,9\},\exists d \in D,\; P(i,j,d)\wedge \forall d' \in D,\bigl(P(i,j,d') \to d' = d\bigr) $$
This condition guarantees both existence and uniqueness: every cell contains a digit, and that digit is uniquely determined.
<br> For the purposes of solving, it is also useful to associate each unsolved cell $(i,j)$ with a candidate set
$$ C(i,j) \subseteq D,$$
containing all digits that do not violate the row, column or the subgrid conditions.

## Strategy definition
This section introduces the strategies used in Sudoku solving, starting with elementary techniques and moving gradually toward more advanced methods that are typically needed only for the most challenging puzzles. **Unit** will be used as a common term for each row, column or $3 \times 3$ subgrid with a size of $9$ cells, as defined above.
<br> Before turning to the main solving strategies, it is useful to first introduce the two most elementary techniques: **Full House** and **Last Digit**.
<br> Full house refers to a unit, for which only one digit is missing.
<br> Last Digit occurs when $8$ of the digit $d$ are placed on the grid. They constrain the grid in such a way that the $9$th digit $d$ can only go in one particular cell.
<br>

### Naked singles
A naked single is found when there is only one candidate for a cell.
$$|C(i,j)| = 1.$$
In other words, if $C(i,j) = \{d\}$, then the cell $(i,j)$ must contain the digit $d$.
<br>

### Hidden singles
A hidden single is a candidate $d \in C(i,j)$ which appears just once in a unit.
<br> Here the question is not "What digits can go in this cell?", rather than "Where can the digit $d$ go in this unit $U$?" We define
$$S_U(d) = \{(i,j) \in U \ | \ d \in C(i,j)\},$$
where $S_U$ denotes the set of cells in unit $U$ in which digit $d$ is a candidate. If $|S_U(d)| = 1,$ digit $d$ can only be placed in cell $(i,j) \in S_U(d).$
<br>

### Naked pairs/triples
A naked **pair** is formed by two cells in the same unit which have the same two (not more) candidates. Let the cells $x_1,x_2 \in U$, digits $a,b \in D,$ $a \neq b.$ Two cells form a naked pair if the candidates $C(x_1) = C(x_2) = \{a,b\}.$ This essentially means that the cells $x_1,x_2$ are reserved for the digits $a,b$, since placement of those digits anywhere in the unit $U$ would force one of the cells $x_1,x_2$ to have no candidates, which leaves a cell empty, which itself violates the fourth constraint described in the former section.
<br> Knowing this, we can take any cell $y \in U \setminus \{x_1,x_2\}$ and update its candidates by removing the digits $a,b$, if present
$$ C(y) \leftarrow C(y) \setminus \{a,b\}. $$
<br>
<br> A naked **triple** is any combination of three cells $x_1, x_2, x_3 \in U$ for which a set $T \subseteq D$ exists such that $$C(x_1) \subseteq T, \qquad C(x_2) \subseteq T, \qquad C(x_3) \subseteq T$$ and $$ C(x_1) \cup C(x_2) \cup C(x_3) = T, \qquad |T| = 3.$$
Since the digits in $T = \{a,b,c\}$ must occupy the cells $x_1,x_2,x_3$, we can safely remove them from the candidates in every cell $y \in U \setminus \{x_1,x_2,x_3\}$
$$ C(y) \leftarrow C(y) \setminus \{a,b,c,\}. $$
<br>

### Hidden pairs/triples
For a hidden **pair** we need to observe the *candidates of cells* $S_U(a), S_U(b)$ in a unit $U$ for two digits $a,b \in D, a \neq b$. If those digits have the same *two* cells as candidates $S_U(a) = S_U(b) = \{x_1,x_2\}$, we can safely say those two cells $x_1,x_2$ are reserved for the digits $a,b$. Thus, we can update their candidates by removing all else except the digits $a,b$, essentially leaving just $\{a,b\}$
$$ C(x_1) \leftarrow \{a,b\}, $$
$$ C(x_2) \leftarrow \{a,b\}. $$
<br>
<br> A hidden **triple** occurs when all candidates of three distinct digits $a,b,c \in D$ in a unit $U$ are limited to three cells $x_1,x_2,x_3.$ The candidates of *cells* of each digit can be defined as subsets or equal sets to the set of $\{x_1,x_2,x_3\}$ and the union od those candidates is exactly the set of those three cells.
$$ S_U(a) \subseteq \{x_1,x_2,x_3\}, \qquad S_U(b) \subseteq \{x_1,x_2,x_3\}, \qquad S_U(c) \subseteq \{x_1,x_2,x_3\}, $$
$$ S_U(a) \cup S_U(b) \cup S_U(c) = \{x_1,x_2,x_3\} $$
Since the digits $a,b,c$ can only go in the cells $\{x_1,x_2,x_3\}$, the candidates of those cells can be updated. Contrary to hidden **pairs**, a cell in a hidden triple does not necessarily have to have all digits $\{a,b,c\}$ as candidates. It would be wrong to update that cell's candidates set by declaring it equal to $\{a,b,c\}$, the correct approach would be to take the intersection of the current candidates and $\{a,b,c\}$.
$$ C(x_1) \leftarrow C(x_1) \cap \{a,b,c\}, $$
$$ C(x_2) \leftarrow C(x_2) \cap \{a,b,c\}, $$
$$ C(x_3) \leftarrow C(x_3) \cap \{a,b,c\}. $$
<br>

### Pointing pairs/triples
The identification of a pointing pair/triple happens within a $3 \times 3$ **box** $B$, for which the third constraint applies. If all *cell* candidates $S_B(d) = \{x \in B \; | \; d \in C(x)\}$ lie in the same row $R_r$ or in the same column $K_k$. Then that digit $d$ can only be placed within the box $B$ and can be removed from anywhere else it appears in that row or column outside of box limits. This can be defined as if $2 \leq |S_B(d)| \leq 3$ and
$$ S_B(d) \subset R_r \qquad \mbox{or} \qquad S_B(d) \subset K_k$$
For every cell $y$ in the row $R_r$ or column $K_k$ that is outisde the box $B$
$$ y \in R_r \setminus B \qquad \mbox{or} \qquad y \in K_k \setminus B $$
removed the digit $d$ from the set of candidates of cell $y$
$$ C(y) \leftarrow C(y) \setminus \{d\} $$
<br>

### Box-line reduction
The box-line reduction technique takes the reverse approach and identifies all *cell* candidates for a digit $d$ in a row $R_r$ or a column $K_k$, if the all of them lie within a box $B$. This asserts that for that row or column, the digit can only in that box, so the digit $d$ can be eliminated from the candidates of the rest of the cells in the box $B$. We defined the *cell* candidates of the row $R_r$ or column $K_k$ for the digit $d$ to be a subset of the cells in box $B$
$$ S_{R_r}(d) \subset B \qquad \mbox{or} \qquad S_{K_k}(d) \subset B. $$
Then for every cell $y$ in box $B$ that does not lie on the row $R_r$ or the column $K_k$
$$ y \in B \setminus R_r \qquad \mbox{or} \qquad y \in B \setminus K_k$$
remove the digit $d$ from the candidates of every cell $y$
$$ C(y) \leftarrow C(y) \setminus \{d\}.$$
<br>

### X-Wing
For simplicity, we will observe a row-based X-Wing, the same analogy can be applied for columns.
<br> The X-Wing pattern is recognized when for a row $r_1$ the cell candidates for a digit $d$ are restricted only to the columns $k_1, k_2$ and there exists a second row $r_2 \neq r_1$, for which the digit $d$ can only go in the same two columns $k_1, k_2$.
This creates a rectangular pattern and ensures that for the columns $k_1, k_2$, the digit $d$ is reserved for either of the rows $r_1, r_2$ in either column. We can define the candidates of *cells* for the rows $r_1, r_2$ as

$$ S_{R_1}(d) = \{(r_1, k_1), (r_1, k_2)\} \qquad \mbox{and} \qquad S_{R_2}(d) = \{(r_2, k_1), (r_2, k_2)\}. $$

This allows us to remove the digit $d$ from the candidates in the columns $k_1, k_2$ for any row $r \notin \{r_1,r_2\}$ :

$$ C((r, k_1) \leftarrow C((r, k_1) \setminus \{d\} \qquad \mbox{and} \qquad C((r, k_2) \leftarrow C((r, k_2) \setminus \{d\}. $$
<br>

### Y-Wing
A Y-Wing (a.k.a. XY-Wing) is a pattern of three cells $p,w_1,w_2$ for which $p$ (the pivot) sees both $w_1, w_2$ (the wings), but the wings do not see each other. Cells "see" each other if they are part of the same unit $U$. Additionally, the candidates of $p,w_1,w_2$ can be defined as
$$ C(p) = \{a,b\}, \qquad C(w_1) = \{a,c\}, \qquad C(w_2) = \{b,c\}, $$
where $a,b,c \in D$ are distinct.
<br> Since our pivot $p$ sees both wings, we can enter a quick "either/or" scenario with the candidates of the pivot $C(p) = \{a,b\}$ because this will directly affect one of the wings. If the value in $p$ is $a$, then that leaves $w_1$ to be $c$, and if $p$ is $b$, then $w_2$ will be $c$. This means that in any case of the pivot $p$, digit $c$ will be in either of $w_1,w_2$. This allows us to look at every cell $y$ that sees **both** wings $w_1,w_2$ and eliminate $c$ from its candidates, if present. We can define the cells $y$ as the intersection of both peer sets for $w_1, w_2$:
$$ \mbox{Peer}(w_1) = \{y \in G \setminus \{w_1\} \; | \: \exists U \; \mbox{unit such that} \; w_1 \in U \; \mbox{and} \; y \in U\}, $$
$$ \mbox{Peer}(w_2) = \{y \in G \setminus \{w_2\} \; | \: \exists U \; \mbox{unit such that} \;  w_2 \in U \; \mbox{and} \; y \in U\}. $$
The common peer set can then be defined as $P(w_1,w_2) = \mbox{Peer}(w_1) \cap \mbox{Peer}(w_1)$ and for every cell $y \in P(w_1,w_2)$ we can remove the digit $c$ from its candidates, if present:
$$ C(y) \leftarrow C(y) \setminus \{c\}.$$
<br>

### Swordfish
We will again be observing just the row-based Swordfish technique for simplicity. Similar to the X-Wing, the Swordfish is looking for a single digit $d$ in rows $r_1,r_2,r_3$ that can only be placed in two or three columns, such that those cells also lie in exactly three columns. This pattern can include anywhere from $6$ to $9$ cells and can be visualized as an unfinished (finished if $9$) $3 \times 3$ pattern.
<br>
![Swordfish-rc2.png](images/Swordfish-rc2.png) ![Swordfish-rc.png](images/Swordfish-rc.png)
<br>
For the row-based Swordfish we can define the set of columns $k$ in row $r$ where the digit $d$ is a candidate as
$$ P_r(d) = \{k \in D \; | \; d \in C(r,k)\}. $$
For each such set we need to constrain for $$ i = 1,2,3 \qquad 2 \leq |P_{r_i}(d)| \leq 3 $$
and the union of those sets to be $$ P_{r_1}(d) \cup P_{r_2}(d) \cup P_{r_3}(d) = \{k_1,k_2,k_3\}. $$
The same logic from X-Wing can be applied and we can confidently say that the digit $d$ has a reserved place in the columns $k_1,k_2,k_3$ at the rows that they intersect with $r_1,r_2,r_3$ and can be safely removed from the candidates of any cell that lies on the columns $k_1,k_2,k_3$ and does **not** lie on the rows
$r_1,r_2,r_3$.
<br>This is technically not the most correct way to describe the cells that should be considered for candidate digit $d$ removal, refer to R5C8 in the first picture, with our current definition it falls under consideration when it should not. However, it does no harm since the candidate digit $d$ is already not present there. We eliminate for every
$$ r \notin \{r_1, r_2, r_3\},\; j \in \{1,2,3\}, \qquad C(r,k_j) \leftarrow C(r,k_j) \setminus \{d\} $$
<br>

### Skyscraper
A column-based Skyscraper refers to a single digit pattern for which exist two columns with exactly two cells where a candidate digit $d$ can go, such that one pair of cells lies on the same row (the ground) the other two cells lie on different rows (roofs). The pattern illustrates two skyscrapers with different height. We will describe the set of candidates of *cells* for the digit $d$ as
$$ S_{K_{k_1}}(d) = \{(r_a,k_1), (r_b,k_1)\} $$
$$ S_{K_{k_2}}(d) = \{(r_a,k_2), (r_c,k_2)\}, $$
where $r_a$ is the shared row (the ground) and $r_b \neq r_c$ are the roofs.
<br>The candidates in the shared row $r_a$ cannot be true together, which means at least one of the candidates in the roofs $u = (r_b,c_1), \; v = (r_c,c_2)$ must be true, they can also both be true. This means that for every cell $y$ that sees both roofs $v,u$, we can safely remove the digit $d$ from its candidates. We can define $y$ as:
$$ \mbox{Peer}(u) = \{y \in G \setminus \{u\} \; | \: \exists U \; \mbox{unit such that} \; u \in U \; \mbox{and} \; y \in U\}, $$
$$ \mbox{Peer}(v) = \{y \in G \setminus \{v\} \; | \: \exists U \; \mbox{unit such that} \;  v \in U \; \mbox{and} \; y \in U\}, $$
$$ y \in P(u,v) = \mbox{Peer}(u) \cap \mbox{Peer}(v). $$
Then for every cell $y$ we can remove the digit $d$ from its candidate set:
$$ C(y) = C(y) \setminus \{d\}. $$
<br>


## Code ##

### Imports

In [47]:
import random
from itertools import combinations
from mpl_toolkits.mplot3d import Axes3D
%matplotlib notebook
import plotly.graph_objects as go
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
from IPython.display import HTML

### Puzzle import and visualization
Imports a random puzzle with difficulty between x and y. "all_puzzles.txt" contains 1000 easy, medium and hard puzzles each, borrowed from github.com/grantm. Please **select** a difficulty you would like to test.
* Easy: < 1.5
* Medium: < 2.5
* Hard: < 5

Manual puzzle input also an option.

In [48]:
def retrieve_puzzle(x,y,filename="all_puzzles.txt"):
    matching_puzzles = []

    with open(filename, "r", encoding="utf-8") as file:
        for entry in file:
            entry = entry.strip()

            parts = entry.split()
            if len(parts) < 3:
                continue

            id = parts[0]
            board_str = parts[1]
            difficulty = float(parts[2])

            if x <= difficulty <= y:
                matching_puzzles.append(board_str)
        if not matching_puzzles:
            print(f"No puzzle found with difficulty between {x} and {y}")
            return None

        return random.choice(matching_puzzles)


In [49]:
# Board visualization

def visualize_board(board):
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-"*21)

        row = ""
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row += "| "

            val = board[r][c] if board[r][c] != 0 else "."
            row += str(val) + " "

        print(row)

In [50]:
# Sudoku import - please choose difficulties HERE

difficulty_from = 1
difficulty_to = 5

sdk = retrieve_puzzle(difficulty_from, difficulty_to)

def string_to_board(s):
    board = []
    for i in range(0,81,9):
        row = [int(x) for x in s[i:i+9]]
        board.append(row)
    return board

initial_board = string_to_board(sdk) # used later for animation purposes
board = string_to_board(sdk)



In [51]:
# Board manual input
#
# board = [
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0],
#     [0,0,0,0,0,0,0,0,0]
# ]
#
# visualize_board(board)

### Solving technique functions
Contains:
* Candidate collection
* Naked single
* Hidden single
* Naked pair/triple
* Hidden pair/triple
* Pointing pair/triple
* Box/line reduction
* X-Wing
* Y_Wing
* Swordfish
* Skyscraper
* Validation checks (checks if a board is solved)
* Solver function (runs all solving technique functions and if any is successful, starts from the first)
* Helper functions


In [52]:
# Candidates functions

def get_candidates(board, row, col):
    if board[row][col] != 0:
        return set()

    digits = set(range(1, 10))

    row_nums = set(board[row])

    col_nums = {board[r][col] for r in range(9)}

    start_row = (row // 3) * 3
    start_col = (col // 3) * 3

    box_nums = {
        board[r][c]
        for r in range(start_row, start_row + 3)
        for c in range(start_col, start_col + 3)
    }

    used = row_nums | col_nums | box_nums
    used.discard(0)

    return digits - used

def get_all_candidates(board):
    candidates = [[set() for _ in range(9)] for _ in range(9)]

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                candidates[row][col] = get_candidates(board, row, col)

    return candidates


In [53]:
# Defying an empty list used later for animation and plotting purposes

events = []


In [54]:
# Naked Singles

def naked_singles(board, candidates):
    progress = False

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0 and len(candidates[row][col]) == 1:
                value = next(iter(candidates[row][col]))
                board[row][col] = value
                print(f'Naked single: placed {value} at cell ({row+1},{col+1})')
                events.append({
                    'type': 'placement',
                    'row': row,
                    'col': col,
                    'digit': {value},
                    'tactic': 'Naked single',
                })
                progress = True

    return progress

In [55]:
# Hidden Singles

def hidden_singles_rows(board, candidates):

    for row in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for col in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden single (row): placed {digit} placed at cell ({r+1},{c+1})')
                events.append({
                    'type': 'placement',
                    'row': r,
                    'col': c,
                    'digit': {digit},
                    'tactic': 'Hidden single',
                })
                return True
    return False

def hidden_singles_columns(board, candidates):

    for col in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for row in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden single (column): placed {digit} placed at cell ({r+1},{c+1})')
                events.append({
                    'type': 'placement',
                    'row': r,
                    'col': c,
                    'digit': {digit},
                    'tactic': 'Hidden single',
                })
                return True
    return False

def hidden_singles_boxes(board, candidates):

    for box_row_start in range(0,9,3):
        for box_col_start in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1, 10)}

            for row in range(box_row_start,box_row_start+3):
                for col in range(box_col_start,box_col_start+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row, col))

            for digit in range(1,10):
                if len(digit_positions[digit]) == 1:
                    r, c = digit_positions[digit][0]
                    board[r][c] = digit
                    print(f'Hidden single (box): placed {digit} placed at cell ({r+1},{c+1})')
                    events.append({
                    'type': 'placement',
                    'row': r,
                    'col': c,
                    'digit': {digit},
                    'tactic': 'Hidden single',
                })
                    return True
    return False

In [56]:
# Helper Functions

def get_row_cells(row):
    return [(row, col) for col in range(9)]

def get_col_cells(col):
    return [(row, col) for row in range(9)]

def get_box_cells(start_row, start_col):
    return [
        (row, col)
        for row in range(start_row, start_row + 3)
        for col in range(start_col, start_col + 3)
    ]

In [57]:
# Naked Pairs

def naked_sets_in_unit(board, candidates, unit_cells, set_size):
    progress = False
    if set_size == 2:               # Helper variable used in the print message
        print_variable = 'pair'
    elif set_size == 3:
        print_variable = 'triple'


    candidate_cells = [
        (row, col)
        for row, col in unit_cells
        if board[row][col] == 0 and 2 <= len(candidates[row][col]) <= set_size
    ]

    for cell_combo in combinations(candidate_cells, set_size):
        combo_digits = set()

        for row, col in cell_combo:
            combo_digits.update(candidates[row][col])

        if len(combo_digits) == set_size:
            combo_cells = set(cell_combo)

            for row, col in unit_cells:
                if (row, col) not in combo_cells and board[row][col] == 0:
                    before = candidates[row][col].copy()
                    candidates[row][col] -= combo_digits

                    if candidates[row][col] != before:
                        cell_combo_print = []    # Helper variable to visualize cells as numbers 1-9 and not indices
                        for x in cell_combo:
                            cell_combo_print.append((x[0]+1, x[1]+1))

                        print(
                            f'Naked {print_variable}: Removed digits {tuple(combo_digits)} from cell {(row+1, col+1)} '
                            f'using cells {tuple(cell_combo_print)}'
                        )
                        events.append({
                            'type': 'elimination',
                            'row': row,
                            'col': col,
                            'digit': combo_digits,
                            'tactic': f'Naked {print_variable}',
                        })
                        progress = True

            if progress:
                return True

    return False

def naked_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if naked_sets_in_unit(board, candidates, get_row_cells(row), 2):
            progress = True

    return progress

def naked_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        if naked_sets_in_unit(board, candidates, get_col_cells(col), 2):
            progress = True

    return progress

def naked_pairs_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if naked_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 2):
                progress = True

    return progress

def naked_pairs(board, candidates):
    if naked_pairs_rows(board, candidates):
        return True
    if naked_pairs_cols(board, candidates):
        return True
    if naked_pairs_boxes(board, candidates):
        return True
    return False

def naked_triples_rows(board, candidates):
    progress = False

    for row in range(9):
        if naked_sets_in_unit(board, candidates, get_row_cells(row), 3):
            progress = True

    return progress

def naked_triples_cols(board, candidates):
    progress = False

    for col in range(9):
        if naked_sets_in_unit(board, candidates, get_col_cells(col), 3):
            progress = True

    return progress

def naked_triples_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if naked_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 3):
                progress = True

    return progress

def naked_triples(board, candidates):
    if naked_triples_rows(board, candidates):
        return True
    if naked_triples_cols(board, candidates):
        return True
    if naked_triples_boxes(board, candidates):
        return True
    return False

In [58]:
# Hidden Pairs/Triples

def hidden_sets_in_unit(board, candidates, unit_cells, set_size):
    progress = False
    if set_size == 2:               # Helper variable used in the print message
        print_variable = 'pairs'
    elif set_size == 3:
        print_variable = 'triples'

    digit_positions = {digit: [] for digit in range(1,10)}

    for row, col in unit_cells:
        if board[row][col] == 0:
            for digit in candidates[row][col]:
                digit_positions[digit].append((row, col))

    candidate_digits = [
        digit for digit in range(1, 10)
        if 1 <= len(digit_positions[digit]) <= set_size
    ]

    for digit_combo in combinations(candidate_digits, set_size):
        combo_set = set(digit_combo)
        combo_cells = set()

        for digit in digit_combo:
            combo_cells.update(digit_positions[digit])

        if len(combo_cells) == set_size:
            for row, col in combo_cells:
                before = candidates[row][col].copy()
                candidates[row][col] &= combo_set

                if candidates[row][col] != before:
                    print(
                        f'Hidden {print_variable}: Remove digits {tuple(combo_set)} from cell {(row+1, col+1)}'
                        #f'from {before} to {candidates[row][col]}'
                    )
                    events.append({
                            'type': 'elimination',
                            'row': row,
                            'col': col,
                            'digit': combo_set,
                            'tactic': f'Hidden {print_variable}',
                        })
                    progress = True

            if progress:
                return True

    return False

def hidden_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_sets_in_unit(board, candidates, get_row_cells(row), 2):
            progress = True

    return progress

def hidden_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        if hidden_sets_in_unit(board, candidates, get_col_cells(col), 2):
            progress = True

    return progress

def hidden_pairs_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 2):
                progress = True

    return progress

def hidden_pairs(board, candidates):
    if hidden_pairs_rows(board, candidates):
        return True
    if hidden_pairs_cols(board, candidates):
        return True
    if hidden_pairs_boxes(board, candidates):
        return True
    return False

def hidden_triples_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_sets_in_unit(board, candidates, get_row_cells(row), 3):
            progress = True

    return progress

def hidden_triples_cols(board, candidates):
    progress = False

    for col in range(9):
        if hidden_sets_in_unit(board, candidates, get_col_cells(col), 3):
            progress = True

    return progress

def hidden_triples_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 3):
                progress = True

    return progress

def hidden_triples(board, candidates):
    if hidden_triples_rows(board, candidates):
        return True
    if hidden_triples_cols(board, candidates):
        return True
    if hidden_triples_boxes(board, candidates):
        return True

    return False

In [59]:
# Pointing Pairs/Triples

def pointing_pairs_triples(board, candidates):

    for r in range(0,9,3):
        for c in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1,10)}

            for row in range(r, r+3):
                for col in range(c, c+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row,col))

            for digit in range(1,10):
                cells = digit_positions[digit]

                # Rows
                if len(cells) == 2 and cells[0][0] == cells[1][0]:
                    target_row = cells[0][0]
                    #print(f'Pointing pair of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                print(f'Pointing pair: {digit}s removed from row {target_row+1}')
                                events.append({
                                    'type': 'elimination',
                                    'row': target_row,
                                    'col': x,
                                    'digit': {digit},
                                    'tactic': f'Pointing pair',
                                })
                                return True

                if len(cells) == 3 and cells[0][0] == cells[1][0] == cells[2][0]:
                    target_row = cells[0][0]
                    #print(f'Pointing triple of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                print(f'Pointing triple: {digit}s removed from row {target_row+1}')
                                events.append({
                                    'type': 'elimination',
                                    'row': target_row,
                                    'col': x,
                                    'digit': {digit},
                                    'tactic': f'Pointing triple',
                                })
                                return True

                # Columns
                if len(cells) == 2 and cells[0][1] == cells[1][1]:
                    target_col = cells[0][1]
                    #print(f'Pointing pair of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                print(f'Pointing pair: {digit}s removed from COL {target_col+1}')
                                events.append({
                                    'type': 'elimination',
                                    'row': x,
                                    'col': target_col,
                                    'digit': {digit},
                                    'tactic': f'Pointing pair',
                                })
                                return True

                if len(cells) == 3 and cells[0][1] == cells[1][1] == cells[2][1]:
                    target_col = cells[0][1]
                    #print(f'Pointing triple of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                print(f'Pointing triple: {digit}s removed from COL {target_col+1}')
                                events.append({
                                    'type': 'elimination',
                                    'row': x,
                                    'col': target_col,
                                    'digit': {digit},
                                    'tactic': f'Pointing triple',
                                })
                                return True

    return False


In [60]:
# Box-Line Reduction

def box_line_reduction_rows(board, candidates):
    for r in range(9):
        digit_positions = {digit: [] for digit in range(1,10)}
        for c in range(9):
            if board[r][c] == 0:
                for digit in candidates[r][c]:
                    digit_positions[digit].append((r,c))

        for digit in range (1,10):
            cells = digit_positions[digit]

            if len(cells) in {2,3}:
                possible_cell_cols = {cells[x][1] for x in range(len(cells))}

                if possible_cell_cols.issubset({0,1,2}) or possible_cell_cols.issubset({3,4,5}) or possible_cell_cols.issubset({6,7,8}):
                    block_row_start = (r // 3) * 3
                    block_col_start = (cells[0][1] // 3) * 3
                    for row in range(block_row_start, block_row_start + 3):
                        for col in range(block_col_start, block_col_start + 3):
                            if row != r and board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(f'Box/line reduction: {digit}s in ROW {r+1}, removed from BOX ({block_row_start+1},{block_col_start+1}) candidates')
                                    events.append({
                                        'type': 'elimination',
                                        'row': row,
                                        'col': col,
                                        'digit': {digit},
                                        'tactic': 'Box/line reduction',
                                    })
                                    return True
    return False

def box_line_reduction_cols(board, candidates):
    for c in range(9):
        digit_positions = {digit: [] for digit in range(1,10)}
        for r in range(9):
            if board[r][c] == 0:
                for digit in candidates[r][c]:
                    digit_positions[digit].append((r,c))

        for digit in range (1,10):
            cells = digit_positions[digit]

            if len(cells) in {2,3}:
                possible_cell_rows = {cells[x][0] for x in range(len(cells))}

                if possible_cell_rows.issubset({0,1,2}) or possible_cell_rows.issubset({3,4,5}) or possible_cell_rows.issubset({6,7,8}):
                    block_row_start = (cells[0][0] // 3) * 3
                    block_col_start = (c // 3) * 3
                    for row in range(block_row_start, block_row_start + 3):
                        for col in range(block_col_start, block_col_start + 3):
                            if col != c and board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(f'Box/line reduction for {digit}s in COL {c+1}, removed from BOX ({block_row_start+1},{block_col_start+1}) candidates')
                                    events.append({
                                        'type': 'elimination',
                                        'row': row,
                                        'col': col,
                                        'digit': {digit},
                                        'tactic': 'Box/line reduction',
                                    })
                                    return True
    return False

def box_line_reduction(board, candidates):
    if box_line_reduction_rows(board, candidates):
        return True

    if box_line_reduction_cols(board, candidates):
        return True

    return False


In [61]:
# X-Wing

def x_wing_rows(board, candidates):
    for digit in range(1, 10):
        row_positions = {}              # key is row number, value is (only) a pair of columns for digit

        for row in range(9):
            cols = []
            for col in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    cols.append(col)

            if len(cols) == 2:
                row_positions[row] = cols

        rows = list(row_positions.keys())

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                r1, r2 = rows[i], rows[j]

                if row_positions[r1] == row_positions[r2]:
                    c1, c2 = row_positions[r1]

                    for row in range(9):
                        if row not in {r1, r2}:
                            for col in [c1, c2]:
                                if board[row][col] == 0:
                                    before = candidates[row][col].copy()
                                    candidates[row][col].discard(digit)

                                    if candidates[row][col] != before:
                                        print(f'X-Wing: removed {digit} from cell ({row+1}, {col+1}) using rows ({r1+1},{r2+1}) and cols ({c1+1},{c2+1})')
                                        events.append({
                                            'type': 'elimination',
                                            'row': row,
                                            'col': col,
                                            'digit': {digit},
                                            'tactic': 'X-Wing',
                                        })
                                        return True

    return False

def x_wing_cols(board, candidates):
    for digit in range(1, 10):
        col_positions = {}

        for col in range(9):
            rows = []
            for row in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    rows.append(row)

            if len(rows) == 2:
                col_positions[col] = rows

        cols = list(col_positions.keys())

        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                c1, c2 = cols[i], cols[j]

                if col_positions[c1] == col_positions[c2]:
                    r1, r2 = col_positions[c1]

                    for col in range(9):
                        if col not in {c1, c2}:
                            for row in [r1, r2]:
                                if board[row][col] == 0:
                                    before = candidates[row][col].copy()
                                    candidates[row][col].discard(digit)

                                    if candidates[row][col] != before:
                                        print(f'X-Wing: removed {digit} from cell ({row+1}, {col+1}) using rows ({r1+1},{r2+1}) and cols ({c1+1},{c2+1})')
                                        events.append({
                                            'type': 'elimination',
                                            'row': row,
                                            'col': col,
                                            'digit': {digit},
                                            'tactic': 'X-Wing',
                                        })
                                        return True

    return False

def x_wing(board, candidates):
    if x_wing_rows(board, candidates):
        return True
    if x_wing_cols(board, candidates):
        return True
    return False


In [62]:
# Y-Wing

def cells_see_each_other(cell1, cell2):
    r1, c1 = cell1
    r2, c2 = cell2

    return (
        r1 == r2 or
        c1 == c2 or
        (r1 // 3 == r2 // 3 and c1 // 3 == c2 // 3)
    )

def get_peers(row, col):
    peers = set()

    for c in range(9):
        if c != col:
            peers.add((row, c))

    for r in range(9):
        if r != row:
            peers.add((r, col))

    box_row_start = (row // 3) * 3
    box_col_start = (col // 3) * 3

    for r in range(box_row_start, box_row_start + 3):
        for c in range(box_col_start, box_col_start + 3):
            if (r, c) != (row, col):
                peers.add((r, c))

    return peers

def y_wing(board, candidates):
    bivalue_cells = []

    # collect all cells with exactly 2 candidates
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0 and len(candidates[row][col]) == 2:
                bivalue_cells.append((row, col))

    # try each bivalue cell as the pivot
    for pivot in bivalue_cells:
        pr, pc = pivot
        pivot_set = candidates[pr][pc]

        # find first wing
        for wing1 in bivalue_cells:
            if wing1 == pivot:
                continue

            if not cells_see_each_other(pivot, wing1):
                continue

            w1r, w1c = wing1
            wing1_set = candidates[w1r][w1c]

            shared1 = pivot_set & wing1_set
            if len(shared1) != 1:
                continue

            # find second wing
            for wing2 in bivalue_cells:
                if wing2 == pivot or wing2 == wing1:
                    continue

                if not cells_see_each_other(pivot, wing2):
                    continue

                w2r, w2c = wing2
                wing2_set = candidates[w2r][w2c]

                shared2 = pivot_set & wing2_set
                if len(shared2) != 1:
                    continue

                # wings must share different pivot digits
                if shared1 == shared2:
                    continue

                # together, pivot + wings must use exactly 3 digits
                union_digits = pivot_set | wing1_set | wing2_set
                if len(union_digits) != 3:
                    continue

                # the digit shared by the two wings but not in the pivot is the elimination digit
                elim_digit_set = (wing1_set & wing2_set) - pivot_set
                if len(elim_digit_set) != 1:
                    continue

                elim_digit = next(iter(elim_digit_set))

                # eliminate from cells that see both wings
                common_peers = get_peers(w1r, w1c) & get_peers(w2r, w2c)

                for row, col in common_peers:
                    if board[row][col] == 0:
                        before = candidates[row][col].copy()
                        candidates[row][col].discard(elim_digit)

                        if candidates[row][col] != before:
                            pivot_print = tuple([x+1 for x in pivot])
                            wing1_print = tuple([x+1 for x in wing1])
                            wing2_print = tuple([z+1 for z in wing2])
                            print(
                                f'Y-Wing: removed {elim_digit} from cell ({row+1}, {col+1}) '
                                f'using pivot {pivot_print}, wing1 {wing1_print}, wing2 {wing2_print}'
                            )
                            events.append({
                                            'type': 'elimination',
                                            'row': row,
                                            'col': col,
                                            'digit': {elim_digit},
                                            'tactic': 'Y-Wing',
                                        })
                            return True

    return False


In [63]:
# Swordfish

def swordfish_rows(board, candidates):
    for digit in range(1, 10):
        row_positions = {}

        for row in range(9):
            cols = []
            for col in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    cols.append(col)

            if 2 <= len(cols) <= 3:
                row_positions[row] = set(cols)

        rows = list(row_positions.keys())

        for row_combo in combinations(rows, 3):
            union_cols = set()
            for row in row_combo:
                union_cols.update(row_positions[row])

            if len(union_cols) == 3:
                for row in range(9):
                    if row not in row_combo:
                        for col in union_cols:
                            if board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    row_combo_print = tuple([x+1 for x in row_combo])
                                    union_cols_print = tuple([x+1 for x in union_cols])

                                    print(
                                        f'Swordfish: removed {digit} from cell ({row+1}, {col+1}) '
                                        f'using rows {row_combo_print} and cols {union_cols_print}'
                                    )
                                    events.append({
                                            'type': 'elimination',
                                            'row': row,
                                            'col': col,
                                            'digit': {digit},
                                            'tactic': 'Swordfish',
                                        })
                                    return True
    return False

def swordfish_cols(board, candidates):
    for digit in range(1, 10):
        col_positions = {}

        for col in range(9):
            rows = []
            for row in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    rows.append(row)

            if 2 <= len(rows) <= 3:
                col_positions[col] = set(rows)

        cols = list(col_positions.keys())

        for col_combo in combinations(cols, 3):
            union_rows = set()
            for col in col_combo:
                union_rows.update(col_positions[col])

            if len(union_rows) == 3:
                for col in range(9):
                    if col not in col_combo:
                        for row in union_rows:
                            if board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    col_combo_print = tuple([x+1 for x in col_combo])
                                    union_rows_print = tuple([x+1 for x in union_rows])

                                    print(
                                        f'Swordfish: removed {digit} from cell ({row+1}, {col+1}) '
                                        f'using cols {col_combo_print} and rows {union_rows_print}'
                                    )
                                    events.append({
                                            'type': 'elimination',
                                            'row': row,
                                            'col': col,
                                            'digit': {digit},
                                            'tactic': 'Swordfish',
                                        })
                                    return True
    return False

def swordfish(board, candidates):
    if swordfish_rows(board, candidates):
        return True
    if swordfish_cols(board, candidates):
        return True
    return False

In [64]:
# Skyscraper

def skyscraper_rows(board, candidates):
    for digit in range(1, 10):
        row_positions = {}

        for row in range(9):
            cols = []
            for col in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    cols.append(col)

            if len(cols) == 2:
                row_positions[row] = set(cols)

        rows = list(row_positions.keys())

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                r1, r2 = rows[i], rows[j]

                common_cols = row_positions[r1] & row_positions[r2]

                if len(common_cols) == 1:
                    shared_col = next(iter(common_cols))
                    roof1_col = next(iter(row_positions[r1] - {shared_col}))
                    roof2_col = next(iter(row_positions[r2] - {shared_col}))

                    roof1 = (r1, roof1_col)
                    roof2 = (r2, roof2_col)

                    common_peers = get_peers(*roof1) & get_peers(*roof2)

                    for row, col in common_peers:
                        if board[row][col] == 0:
                            before = candidates[row][col].copy()
                            candidates[row][col].discard(digit)

                            if candidates[row][col] != before:
                                roof1_print = tuple([x+1 for x in roof1])
                                roof2_print = tuple([x+1 for x in roof2])

                                print(
                                    f'Skyscraper: removed {digit} from ({row+1}, {col+1}) '
                                    f'using rows ({r1+1},{r2+1}) with shared col {shared_col+1} '
                                    f'and roofs ({roof1_print}, {roof2_print})'
                                )
                                events.append({
                                            'type': 'elimination',
                                            'row': row,
                                            'col': col,
                                            'digit': {digit},
                                            'tactic': 'Skyscraper',
                                        })
                                return True
    return False

def skyscraper_cols(board, candidates):
    for digit in range(1, 10):
        col_positions = {}

        for col in range(9):
            rows = []
            for row in range(9):
                if board[row][col] == 0 and digit in candidates[row][col]:
                    rows.append(row)

            if len(rows) == 2:
                col_positions[col] = set(rows)

        cols = list(col_positions.keys())

        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                c1, c2 = cols[i], cols[j]

                common_rows = col_positions[c1] & col_positions[c2]

                if len(common_rows) == 1:
                    shared_row = next(iter(common_rows))
                    roof1_row = next(iter(col_positions[c1] - {shared_row}))
                    roof2_row = next(iter(col_positions[c2] - {shared_row}))

                    roof1 = (roof1_row, c1)
                    roof2 = (roof2_row, c2)

                    common_peers = get_peers(*roof1) & get_peers(*roof2)

                    for row, col in common_peers:
                        if board[row][col] == 0:
                            before = candidates[row][col].copy()
                            candidates[row][col].discard(digit)

                            if candidates[row][col] != before:
                                roof1_print = tuple([x+1 for x in roof1])
                                roof2_print = tuple([x+1 for x in roof2])

                                print(
                                    f'Skyscraper: removed {digit} from ({row+1}, {col+1}) '
                                    f'using cols {c1+1},{c2+1} with shared row {shared_row+1} '
                                    f'and roofs {roof1_print}, {roof2_print}'
                                )
                                events.append({
                                            'type': 'elimination',
                                            'row': row,
                                            'col': col,
                                            'digit': {digit},
                                            'tactic': 'Skyscraper',
                                        })
                                return True
    return False

def skyscraper(board, candidates):
    if skyscraper_rows(board, candidates):
        return True
    if skyscraper_cols(board, candidates):
        return True
    return False

In [65]:
# Validation checks

def is_valid_row(board, row):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for cell in range(9):
        digits.add(board[row][cell])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_col(board, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()
    for cell in range(9):
        digits.add(board[cell][col])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_block(board, row, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for r in range(row, row+3):
        for c in range(col, col+3):
            digits.add(board[r][c])

    if digits == desired_set:
        return True
    else:
        return False

def is_solved(board):
    errors = 0
    for row in range(9):
        is_valid = is_valid_row(board, row)
        if not is_valid:
            errors += 1
            #print(f'Board not valid at row {row+1}')

    for col in range(9):
        is_valid = is_valid_col(board, col)
        if not is_valid:
            errors += 1
            #print(f'Board not valid at col {col+1}')

    for row in range(0,9,3):
        for col in range(0,9,3):
            is_valid = is_valid_block(board, row, col)
            if not is_valid:
                errors += 1
                #print(f'Board not valid at block ({row+1}, {col+1})')

    if errors == 0:
        return True
    else:
        return False


In [66]:
# Main solver

def run_solver(board):
    candidates = get_all_candidates(board)
    while True:

        if naked_singles(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_rows(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_columns(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_boxes(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if naked_pairs(board, candidates):
            continue

        if naked_triples(board, candidates):
            continue

        if hidden_pairs(board, candidates):
            continue

        if hidden_triples(board, candidates):
            continue

        if pointing_pairs_triples(board, candidates):
            continue

        if box_line_reduction(board, candidates):
            continue

        if x_wing(board, candidates):
            continue

        if y_wing(board, candidates):
            continue

        if swordfish(board, candidates):
            continue

        if skyscraper(board, candidates):
            continue

        if is_solved(board):
            print('Board is solved, great success :)')

        else:
            print("No more progress can be made, too hard :(")

        visualize_board(board)
        break


### Execution
What to expect:
1. **Main solver**: a list of consecutive prints of strategies and what and where they changed on the board and the board itself in its final form.
2. **Animation**: An animated video of the puzzle being solved in real time. Initial digits will be black, all new digits blue. Red highlighted cell means a placement of a digit, blue highlight means candidate removal. The step, placement or elimination, cell, digit and tactic stay on top of the board and change with each iteration. Expect the animation to load ~30s.
3. **3D Solving path**: A 3D plot with each cell placement/candidate elimination mapped. Row, column and step are mapped on the x,y and z axis. Red circle again means placement, blue diamond means elimination. All pointns are connected with a line. Each point, when hovered above, shows step, row, column, digit, placement or elimination, tactic. Tweak the steps_ratio if z axis is too stretched or compacted.

In [67]:
# Main solver run

run_solver(board)

Naked single: placed 5 at cell (7,3)
Naked single: placed 2 at cell (8,4)
Hidden single (row): placed 3 placed at cell (4,3)
Hidden single (row): placed 6 placed at cell (5,7)
Hidden single (row): placed 6 placed at cell (4,5)
Hidden single (row): placed 6 placed at cell (7,2)
Hidden single (row): placed 5 placed at cell (8,9)
Naked single: placed 4 at cell (2,9)
Hidden single (row): placed 4 placed at cell (3,1)
Hidden single (row): placed 6 placed at cell (3,9)
Hidden single (row): placed 6 placed at cell (1,1)
Hidden single (row): placed 4 placed at cell (6,8)
Hidden single (row): placed 4 placed at cell (5,6)
Hidden single (row): placed 8 placed at cell (8,1)
Hidden single (row): placed 8 placed at cell (4,4)
Naked single: placed 7 at cell (5,4)
Hidden single (row): placed 8 placed at cell (5,3)
Hidden single (row): placed 4 placed at cell (9,3)
Hidden single (column): placed 8 placed at cell (3,5)
Hidden single (column): placed 2 placed at cell (2,5)
Naked single: placed 7 at cell

In [68]:
# Animating the solution path with candidate removal and digit placing

def animate_sudoku_with_candidates(events, initial_board, interval=700):
    board = [row[:] for row in initial_board]
    candidates = get_all_candidates(board)

    # track cells that were filled by the solver, so they stay blue
    placed_cells = set()

    fig, ax = plt.subplots(figsize=(8, 8))

    def draw_candidates_in_cell(row, col):
        """
        Draw pencil marks inside one empty cell using a 3x3 mini-grid.
        """
        cell_candidates = candidates[row][col]

        for digit in range(1, 10):
            if digit in cell_candidates:
                mini_row = (digit - 1) // 3
                mini_col = (digit - 1) % 3

                x = col + 0.17 + mini_col * 0.33
                y = row + 0.22 + mini_row * 0.28

                ax.text(
                    x, y, str(digit),
                    ha="center", va="center",
                    fontsize=7, color="gray"
                )

    def draw_board(highlight=None, info_text=""):
        ax.clear()
        ax.set_xlim(0, 9)
        ax.set_ylim(0, 9)
        ax.invert_yaxis()
        ax.set_xticks([])
        ax.set_yticks([])

        # draw cells
        for row in range(9):
            for col in range(9):
                color = "white"

                if highlight and (row, col) == (highlight["row"], highlight["col"]):
                    if highlight["type"] == "placement":
                        color = "lightcoral"
                    else:
                        color = "lightblue"

                rect = Rectangle(
                    (col, row), 1, 1,
                    facecolor=color,
                    edgecolor="black",
                    linewidth=1
                )
                ax.add_patch(rect)

                # draw big solved digit
                if board[row][col] != 0:
                    text_color = "blue" if (row, col) in placed_cells else "black"

                    ax.text(
                        col + 0.5, row + 0.58, str(board[row][col]),
                        ha="center", va="center",
                        fontsize=16,
                        color=text_color
                    )
                else:
                    draw_candidates_in_cell(row, col)

        # draw thick 3x3 box borders
        for i in range(10):
            lw = 2.5 if i % 3 == 0 else 1
            ax.plot([i, i], [0, 9], color="black", linewidth=lw)
            ax.plot([0, 9], [i, i], color="black", linewidth=lw)

        ax.set_title(info_text)

    def update(frame):
        event = events[frame]


        digits = set(event["digit"])

        row = event["row"]
        col = event["col"]

        # apply event to internal state
        if event["type"] == "placement":
            value = next(iter(digits))
            board[row][col] = value
            candidates[row][col] = set()
            placed_cells.add((row, col))

        elif event["type"] == "elimination":
            if board[row][col] == 0:
                candidates[row][col] -= digits

        digit_text = ",".join(str(d) for d in sorted(digits))

        info = (
            f"Step {frame + 1} | "
            f"{event['type'].capitalize()} | "
            f"Cell ({row + 1},{col + 1}) | "
            f"Digit(s): {digit_text} | "
            f"Tactic: {event['tactic']}"
        )

        draw_board(highlight=event, info_text=info)

    anim = FuncAnimation(
        fig,
        update,
        frames=len(events),
        interval=interval,
        repeat=False
    )

    return anim

In [69]:
# Animation run

anim = animate_sudoku_with_candidates(events, initial_board)
HTML(anim.to_jshtml())

<IPython.core.display.Javascript object>

In [70]:
# 3D solving path visualization (change 'steps_ratio' to stretch the z axis by taste)

def plot_solution_path_3d(events, steps_ratio = 1.6):
    if not events:
        print("No events to plot.")
        return

    xs = [event["col"] + 1 for event in events]
    ys = [event["row"] + 1 for event in events]
    zs = [(i + 1) for i in range(len(events))]

    colors = [
        "red" if event["type"] == "placement" else "blue"
        for event in events
    ]

    symbols = [
        "circle" if event["type"] == "placement" else "diamond"
        for event in events
    ]

    labels = []
    hover_texts = []

    for i, event in enumerate(events):
        digit_value = event["digit"]

        if isinstance(digit_value, set):
            label = ",".join(str(d) for d in sorted(digit_value))
        else:
            label = str(digit_value)

        labels.append(label)

        hover_texts.append(
            f"Step: {i+1}<br>"
            f"Row: {event['row']+1}<br>"
            f"Col: {event['col']+1}<br>"
            f"Digit(s): {label}<br>"
            f"Type: {event['type']}<br>"
            f"Tactic: {event['tactic']}"
        )

    fig = go.Figure()

    # line connecting the events
    fig.add_trace(go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode="lines",
        line=dict(color="gray", width=4),
        hoverinfo="skip",
        showlegend=False
    ))

    # points
    fig.add_trace(go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode="markers+text",
        marker=dict(
            size=6,
            color=colors,
            symbol=symbols
        ),
        text=labels,
        textposition="top center",
        hovertext=hover_texts,
        hoverinfo="text",
        name="Events"
    ))

    fig.update_layout(
        title="Sudoku 3D Solution Path",
        scene=dict(
            xaxis=dict(title="Column", range=[1, 9]),
            yaxis=dict(title="Row", range=[1, 9]),
            zaxis=dict(title="Step", range=[1, len(events)]),
            aspectratio=dict(x=1, y=1, z=steps_ratio),
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

In [71]:
# Plot run

plot_solution_path_3d(events)

## References

1. grantm. *sudoku-exchange-puzzle-bank*. GitHub repository. Available at: https://github.com/grantm/sudoku-exchange-puzzle-bank/tree/master
2. Sudopedia. *Solving Technique*. Available at: https://www.sudopedia.org/wiki/Solving_Technique
3. omerfarukeker. *Sudoku-Solver*. GitHub repository. Available at: https://github.com/omerfarukeker/Sudoku-Solver
4. Lin, Ren. *Sudoku as Sets and Predicate Logic*. Used for the formalization of Sudoku in terms of sets, predicates, relations, and satisfiability.
5. Stoll, Robert R. *Set Theory and Logic*. Dover Publications, 1979. Referenced in Lin’s paper.